# Semantic Router for Intent Classification

This notebook demonstrates using `SemanticRouter` to classify user queries into predefined routes based on semantic similarity.

**Use Cases:**
- Route queries to specialized agents
- Implement intent detection for chatbots
- Classify support tickets by topic
- Direct users to appropriate resources

**Built on:** RedisVL's SemanticRouter with vector similarity search

## Setup

Import dependencies and configure environment.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Redis connection
REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Environment configured")
print(f"   Redis URL: {REDIS_URL}")

In [ ]:
from redis_openai_agents import SemanticRouter, Route, RouteMatch

print("SemanticRouter imported successfully")

## Define Routes

Create routes with reference phrases that exemplify each category. The router will classify queries based on semantic similarity to these references.

In [ ]:
# Define routes for a multi-topic assistant
technology = Route(
    name="technology",
    references=[
        "What are the latest advancements in AI?",
        "Tell me about new programming languages",
        "How does machine learning work?",
        "What is cloud computing?",
        "Explain blockchain technology",
    ],
    metadata={"department": "engineering", "priority": "normal"},
    distance_threshold=0.7,
)

sports = Route(
    name="sports",
    references=[
        "Who won the football game last night?",
        "What are the NBA standings?",
        "Tell me about the World Cup",
        "Best tennis players of all time",
        "Olympic medal counts",
    ],
    metadata={"department": "entertainment", "priority": "low"},
    distance_threshold=0.7,
)

support = Route(
    name="support",
    references=[
        "I need help with my account",
        "How do I reset my password?",
        "I'm having trouble logging in",
        "Where is my order?",
        "I want to cancel my subscription",
    ],
    metadata={"department": "customer_service", "priority": "high"},
    distance_threshold=0.6,
)

print("Routes defined:")
print(f"   - technology: {len(technology.references)} references")
print(f"   - sports: {len(sports.references)} references")
print(f"   - support: {len(support.references)} references")

## Create Router

Initialize the semantic router with our defined routes.

In [ ]:
router = SemanticRouter(
    name="topic-router",
    routes=[technology, sports, support],
    redis_url=REDIS_URL,
    overwrite=True,  # Replace existing index if it exists
)

print("SemanticRouter created")
print(f"   Name: {router.name}")
print(f"   Routes: {router.route_names}")

## Basic Routing

Classify individual queries to find the best matching route.

In [ ]:
# Test with a technology query
query = "Can you explain how neural networks work?"
result = await router(query)

print(f"Query: {query}")
print(f"Route: {result.name}")
print(f"Distance: {result.distance:.4f}")

In [ ]:
# Test with a sports query
query = "Who is the best basketball player?"
result = await router(query)

print(f"Query: {query}")
print(f"Route: {result.name}")
print(f"Distance: {result.distance:.4f}")

In [ ]:
# Test with a support query
query = "I can't access my account"
result = await router(query)

print(f"Query: {query}")
print(f"Route: {result.name}")
print(f"Distance: {result.distance:.4f}")

## Multiple Matches with route_many

Get multiple potential routes ranked by similarity. Useful when queries might belong to multiple categories.

In [ ]:
# Query that might match multiple routes
query = "Technical help with my sports streaming app"
results = await router.route_many(query, max_k=3)

print(f"Query: {query}")
print(f"\nMatching routes (best to worst):")
for i, match in enumerate(results, 1):
    print(f"   {i}. {match.name} (distance: {match.distance:.4f})")

## Aggregation Methods

When a route has multiple references, you can choose how to combine their distances:
- **avg**: Average distance across all references (default)
- **min**: Use the minimum distance (closest match)
- **sum**: Sum of all distances

In [ ]:
query = "What are the latest programming trends?"

# Try different aggregation methods
for method in ["avg", "min"]:
    result = await router(query, aggregation_method=method)
    print(f"{method.upper()}: route={result.name}, distance={result.distance:.4f}")

## Route Metadata

Access metadata associated with routes for additional context.

In [ ]:
# Route a query and get associated metadata
query = "I need to fix my billing issue"
result = await router(query)

if result.name:
    route = router.get_route(result.name)
    print(f"Query: {query}")
    print(f"Routed to: {result.name}")
    print(f"Metadata: {route.metadata}")
    print(f"\nThis query should be handled by: {route.metadata.get('department')}")
    print(f"Priority: {route.metadata.get('priority')}")

## Managing Route References

Add new references to improve route matching over time.

In [ ]:
# Get current references for a route
refs = await router.get_route_references("technology")
print(f"Current technology references: {len(refs)}")

# Add new references
await router.add_route_references(
    "technology",
    [
        "What is quantum computing?",
        "How do self-driving cars work?",
    ]
)

# Verify
refs = await router.get_route_references("technology")
print(f"After adding: {len(refs)} references")

In [ ]:
# Test with newly added reference topic
query = "Explain autonomous vehicles"
result = await router(query)

print(f"Query: {query}")
print(f"Route: {result.name}")
print(f"Distance: {result.distance:.4f}")

## Integration with OpenAI Agents

Use the router to direct queries to specialized agents.

In [ ]:
from agents import Agent, Runner

# Create specialized agents
tech_agent = Agent(
    name="TechExpert",
    instructions="You are a technology expert. Explain technical concepts clearly."
)

sports_agent = Agent(
    name="SportsAnalyst",
    instructions="You are a sports analyst. Provide insightful sports commentary."
)

support_agent = Agent(
    name="SupportRep",
    instructions="You are a helpful customer support representative. Resolve issues efficiently."
)

# Map routes to agents
agents = {
    "technology": tech_agent,
    "sports": sports_agent,
    "support": support_agent,
}

print("Specialized agents created")

In [ ]:
async def route_and_respond(query: str) -> str:
    """Route query to appropriate agent and get response."""
    # Classify the query
    result = await router(query)
    
    if result.name is None:
        return "I'm not sure how to help with that."
    
    # Get the appropriate agent
    agent = agents.get(result.name)
    if not agent:
        return f"No agent configured for route: {result.name}"
    
    print(f"[Router] Query: {query}")
    print(f"[Router] Route: {result.name} (distance: {result.distance:.4f})")
    print(f"[Router] Forwarding to: {agent.name}")
    print()
    
    # Get response from the specialized agent
    response = await Runner.run(agent, input=query)
    return response.final_output

In [ ]:
# Test the routing system
queries = [
    "What is the difference between AI and machine learning?",
    "Who won the Super Bowl this year?",
    "I need to update my payment method",
]

for query in queries:
    response = await route_and_respond(query)
    print(f"Response: {response[:150]}..." if len(response) > 150 else f"Response: {response}")
    print("-" * 80)

## Serialization

Save and load router configuration.

In [ ]:
# Export router config
config = router.to_dict()
print("Router configuration:")
print(f"   Name: {config['name']}")
print(f"   Routes: {len(config['routes'])}")
for route in config['routes']:
    print(f"      - {route['name']}: {len(route['references'])} refs, threshold={route['distance_threshold']}")

In [ ]:
# Create a new router from config
restored_router = await SemanticRouter.from_dict(
    config,
    redis_url=REDIS_URL,
    overwrite=True,
)

# Test it works
result = await restored_router("Tell me about cloud services")
print(f"Restored router works: {result.name}")
await restored_router.close()

## Distance Threshold Tuning

Adjust thresholds to control matching sensitivity.

In [ ]:
# Create routes with different thresholds
strict_route = Route(
    name="strict",
    references=["very specific phrase about databases"],
    distance_threshold=0.3,  # Very strict - only close matches
)

lenient_route = Route(
    name="lenient",
    references=["general question about technology"],
    distance_threshold=1.0,  # Lenient - broader matching
)

threshold_router = SemanticRouter(
    name="threshold-demo",
    routes=[strict_route, lenient_route],
    redis_url=REDIS_URL,
    overwrite=True,
)

# Test with various queries
test_queries = [
    "very specific phrase about databases",  # Exact match
    "tell me about SQL databases",           # Related but different
    "something about computers",             # Vaguely related
]

print("Testing threshold behavior:")
for q in test_queries:
    result = await threshold_router(q)
    print(f"   Query: {q[:40]:40} -> {result.name or 'NO MATCH'} (d={result.distance or 0:.3f})")

await threshold_router.close()

## Cleanup

In [ ]:
# Close the router
await router.close()
print("Router closed")

## Summary

This notebook demonstrated:

1. **Route Definition** - Create routes with reference phrases and metadata
2. **Basic Routing** - Classify queries to the best matching route
3. **Multiple Matches** - Get ranked list of potential routes
4. **Aggregation** - Control how reference distances are combined
5. **Reference Management** - Add references to improve matching
6. **Agent Integration** - Route queries to specialized agents
7. **Serialization** - Save and restore router configuration
8. **Threshold Tuning** - Control matching sensitivity

**Key Benefits:**
- Route queries to specialized agents/handlers
- Semantic matching handles paraphrases naturally
- Metadata enables downstream processing
- Easy to extend with new references

**Best Practices:**
- Include 5-10 diverse references per route
- Set stricter thresholds for critical routes
- Use route_many when queries might span categories
- Add references based on actual user queries over time